# Gated Fusion Experiment: RGB + Depth + Joint Encoders

## Objective
This notebook implements a **multimodal gated fusion model** for posture classification using three pretrained encoders:

- **RGB encoder**
- **Depth encoder**
- **Joint encoder**

The goal is to combine feature representations from all three modalities and train a **gated fusion classification head** on top of them.

---

## Experiment Setup
In this experiment:

- pretrained encoder weights are loaded from saved artifacts
- encoder backbones are used as **feature extractors**
- encoder parameters are initially **frozen**
- a **gated fusion module** is trained to combine modality-specific features
- the final classifier predicts one of the posture classes:
  - **supine**
  - **left**
  - **right**

---

## Why Gated Fusion
Gated fusion allows the model to learn the **relative importance of each modality** for a given sample.

This is useful because:

- **RGB** may become less reliable under blanket occlusion
- **Depth** may retain useful spatial posture information
- **Joint features** may provide structural cues but may also be noisy

The gating mechanism helps the model dynamically weight each modality before final classification.

## Imports and Setup

In [15]:
# =========================
# Core
# =========================
from pathlib import Path
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import numpy as np
import random
import scipy.io as sio
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# =========================
# PyTorch
# =========================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models

In [2]:
# =========================
# Device setup
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Paths and Configuration

In [3]:
# =========================
# Paths
# =========================
PROJECT_ROOT = Path.cwd().resolve().parent

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"

DEPTH_ARTIFACT_DIR = ARTIFACTS_DIR / "depth_encoder_cnn" / "checkpoints"
RGB_ARTIFACT_DIR = ARTIFACTS_DIR / "rgb_encoder_cnn" / "checkpoints"
JOINT_ARTIFACT_DIR = ARTIFACTS_DIR / "joint_xyo" / "checkpoints"

FUSION_CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints_fusion_gated"
FUSION_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Depth artifacts:", DEPTH_ARTIFACT_DIR)
print("RGB artifacts:", RGB_ARTIFACT_DIR)
print("Joint artifacts:", JOINT_ARTIFACT_DIR)
print("Fusion checkpoints:", FUSION_CHECKPOINT_DIR)

Project root: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense
Depth artifacts: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\depth_encoder_cnn\checkpoints
RGB artifacts: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\rgb_encoder_cnn\checkpoints
Joint artifacts: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\joint_xyo\checkpoints
Fusion checkpoints: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\checkpoints_fusion_gated


## Experiment Configuration

Define the basic experiment settings for gated fusion, including:

- number of classes
- batch size
- learning rate
- number of epochs
- common feature dimension for fusion

These can be adjusted later if needed.

In [4]:
# =========================
# Experiment config
# =========================
NUM_CLASSES = 3

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 15

COMMON_FEATURE_DIM = 256 # to convert all features from encoders to a common size
DROPOUT = 0.3

CLASS_NAMES = ["supine", "left", "right"]

## Encoder Checkpoint Paths

In [5]:
# =========================
# Encoder checkpoint files
# =========================
DEPTH_CHECKPOINT_PATH = DEPTH_ARTIFACT_DIR / "best_depth_encoder_only.pth"
RGB_CHECKPOINT_PATH = RGB_ARTIFACT_DIR / "best_rgb_encoder.pt"
JOINT_CHECKPOINT_PATH = JOINT_ARTIFACT_DIR / "joint_encoder_xyo_RGB.pth"

print("Depth checkpoint:", DEPTH_CHECKPOINT_PATH)
print("RGB checkpoint:", RGB_CHECKPOINT_PATH)
print("Joint checkpoint:", JOINT_CHECKPOINT_PATH)

Depth checkpoint: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\depth_encoder_cnn\checkpoints\best_depth_encoder_only.pth
RGB checkpoint: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\rgb_encoder_cnn\checkpoints\best_rgb_encoder.pt
Joint checkpoint: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\joint_xyo\checkpoints\joint_encoder_xyo_RGB.pth


## Encoder Model Definitions

Define the encoder architectures needed to load the pretrained weights.

At this stage, we only recreate the model structures required for:

- RGB encoder
- Depth encoder
- Joint encoder

These definitions should match the architectures used during the original encoder training notebooks.

In [6]:
# =========================
# Encoder feature dimensions
# =========================
DEPTH_FEATURE_DIM = 512
RGB_FEATURE_DIM = 128
JOINT_INPUT_DIM = 42
JOINT_FEATURE_DIM = 128

In [7]:
class DepthEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(weights=None)

        # change first conv to 1 channel
        backbone.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        # remove FC
        layers = list(backbone.children())[:-1]

        self.encoder = nn.Sequential(*layers)
        self.flatten = nn.Flatten()

        self.feature_dim = 512

    def forward(self, x):
        x = self.encoder(x)
        x = self.flatten(x)
        return x


class RGBEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(weights=None)
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.projection = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128)
        )
        self.feature_dim = 128

    def forward(self, x):
        x = self.backbone(x)
        x = self.projection(x)
        return x


class JointEncoder(nn.Module):
    def __init__(self, input_dim=42, hidden_dim=128, feature_dim=128, dropout=0.3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, feature_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.feature_dim = feature_dim

    def forward(self, x):
        return self.encoder(x)

## Load Pretrained Encoders

Instantiate the three encoder models and load their encoder-only weights from the saved checkpoints.

After loading:
- move models to the selected device
- set them to evaluation mode
- freeze parameters so that only the gated fusion head is trained in this first experiment stage

In [8]:
# =========================
# Instantiate encoders
# =========================
depth_encoder = DepthEncoder().to(device)
rgb_encoder = RGBEncoder().to(device)
joint_encoder = JointEncoder(
    input_dim=JOINT_INPUT_DIM,
    hidden_dim=128,
    feature_dim=JOINT_FEATURE_DIM,
    dropout=0.3
).to(device)

# =========================
# Load checkpoints
# =========================
depth_ckpt = torch.load(DEPTH_CHECKPOINT_PATH, map_location=device)
rgb_ckpt = torch.load(RGB_CHECKPOINT_PATH, map_location=device)
joint_ckpt = torch.load(JOINT_CHECKPOINT_PATH, map_location=device)

depth_encoder.encoder.load_state_dict(depth_ckpt["encoder_state_dict"])
rgb_encoder.load_state_dict(rgb_ckpt["encoder_state_dict"])
joint_encoder.encoder.load_state_dict(joint_ckpt["encoder_state_dict"])

# =========================
# Freeze encoders
# =========================
for model in [depth_encoder, rgb_encoder, joint_encoder]:
    for param in model.parameters():
        param.requires_grad = False
    model.eval()

print("All encoders loaded and frozen successfully.")

All encoders loaded and frozen successfully.


## Multimodal Dataset and Dataloaders

Create a dataset that returns aligned inputs for all three modalities for the same sample:

- RGB image
- Depth image
- Joint features
- posture label

This ensures the gated fusion model receives synchronized modality inputs during training and evaluation.

In [9]:
# =========================
# Multimodal metadata, dataset, and dataloaders
# =========================

DATASET_ROOT = PROJECT_ROOT.parent / "Dataset" / "SLP2022" / "SLP" / "danaLab"
CSV_PATH = DATASET_ROOT / "posture_labels_all_modalities.csv"

IMG_W = 576.0
IMG_H = 1024.0


def build_fusion_metadata(csv_path):
    df = pd.read_csv(csv_path)
    df["subject_id"] = df["subject_id"].astype(int).astype(str).str.zfill(5)

    # Use RGB rows as the aligned base sample definition
    rgb_df = df[df["modality"] == "RGB"].copy()

    rgb_df["rgb_path"] = (
        rgb_df["subject_id"]
        + "/RGB/"
        + rgb_df["condition"]
        + "/image_"
        + rgb_df["image_index"].astype(int).astype(str).str.zfill(6)
        + ".png"
    )

    rgb_df["depth_path"] = (
        rgb_df["subject_id"]
        + "/depth/"
        + rgb_df["condition"]
        + "/image_"
        + rgb_df["image_index"].astype(int).astype(str).str.zfill(6)
        + ".png"
    )

    # Joints are shared at subject level
    rgb_df["joint_file"] = rgb_df["subject_id"] + "/joints_gt_RGB.mat"
    rgb_df["frame_idx_0based"] = rgb_df["image_index"].astype(int) - 1

    return rgb_df[
        [
            "subject_id",
            "condition",
            "image_index",
            "label",
            "label_id",
            "rgb_path",
            "depth_path",
            "joint_file",
            "frame_idx_0based",
        ]
    ].reset_index(drop=True)


def preprocess_joint_frame_xyo(frame_joints):
    x = frame_joints[0].astype(np.float32) / IMG_W
    y = frame_joints[1].astype(np.float32) / IMG_H
    occ = frame_joints[2].astype(np.float32)
    out = np.stack([x, y, occ], axis=1)
    return out.reshape(-1)


rgb_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

depth_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


class FusionDataset(Dataset):
    def __init__(self, df, dataset_root, rgb_transform=None, depth_transform=None):
        self.df = df.reset_index(drop=True)
        self.dataset_root = Path(dataset_root)
        self.rgb_transform = rgb_transform
        self.depth_transform = depth_transform
        self.joint_cache = {}

    def __len__(self):
        return len(self.df)

    def _load_joint_file(self, joint_rel_path):
        if joint_rel_path not in self.joint_cache:
            mat = sio.loadmat(self.dataset_root / joint_rel_path)
            self.joint_cache[joint_rel_path] = mat["joints_gt"]
        return self.joint_cache[joint_rel_path]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        rgb_img = Image.open(self.dataset_root / row["rgb_path"]).convert("RGB")
        if self.rgb_transform is not None:
            rgb_img = self.rgb_transform(rgb_img)

        depth_img = Image.open(self.dataset_root / row["depth_path"]).convert("L")
        if self.depth_transform is not None:
            depth_img = self.depth_transform(depth_img)

        joints_all = self._load_joint_file(row["joint_file"])
        frame = joints_all[:, :, int(row["frame_idx_0based"])]
        joint_vec = preprocess_joint_frame_xyo(frame)

        return {
            "rgb": rgb_img,
            "depth": depth_img,
            "joint": torch.tensor(joint_vec, dtype=torch.float32),
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "condition": row["condition"],
            "subject_id": row["subject_id"],
        }


def subject_wise_split(subject_ids, train_ratio=0.70, val_ratio=0.15, seed=42):
    subject_ids = sorted(subject_ids)
    rng = random.Random(seed)
    rng.shuffle(subject_ids)

    n = len(subject_ids)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_subjects = subject_ids[:n_train]
    val_subjects = subject_ids[n_train:n_train + n_val]
    test_subjects = subject_ids[n_train + n_val:]

    return train_subjects, val_subjects, test_subjects


fusion_df = build_fusion_metadata(CSV_PATH)

subjects = sorted(fusion_df["subject_id"].unique().tolist())
train_subjects, val_subjects, test_subjects = subject_wise_split(
    subjects,
    train_ratio=0.70,
    val_ratio=0.15,
    seed=42,
)

train_df = fusion_df[fusion_df["subject_id"].isin(train_subjects)].reset_index(drop=True)
val_df = fusion_df[fusion_df["subject_id"].isin(val_subjects)].reset_index(drop=True)
test_df = fusion_df[fusion_df["subject_id"].isin(test_subjects)].reset_index(drop=True)

train_dataset = FusionDataset(
    train_df,
    dataset_root=DATASET_ROOT,
    rgb_transform=rgb_transform,
    depth_transform=depth_transform,
)

val_dataset = FusionDataset(
    val_df,
    dataset_root=DATASET_ROOT,
    rgb_transform=rgb_transform,
    depth_transform=depth_transform,
)

test_dataset = FusionDataset(
    test_df,
    dataset_root=DATASET_ROOT,
    rgb_transform=rgb_transform,
    depth_transform=depth_transform,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Fusion metadata shape:", fusion_df.shape)
print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Fusion metadata shape: (13770, 9)
Train samples: 9585
Val samples: 2025
Test samples: 2160


## Gated Fusion Model

Define a gated fusion classifier that takes feature vectors from:

- depth encoder
- RGB encoder
- joint encoder

Each modality is first projected to a common feature dimension. A learnable gate is then used to reweight each modality before concatenation and final classification.

In [10]:
class GatedFusionModel(nn.Module):
    def __init__(
        self,
        depth_dim=DEPTH_FEATURE_DIM,
        rgb_dim=RGB_FEATURE_DIM,
        joint_dim=JOINT_FEATURE_DIM,
        common_dim=COMMON_FEATURE_DIM,
        num_classes=NUM_CLASSES,
        dropout=DROPOUT
    ):
        super().__init__()

        # Project all modalities to common dimension
        self.depth_proj = nn.Linear(depth_dim, common_dim)
        self.rgb_proj = nn.Linear(rgb_dim, common_dim)
        self.joint_proj = nn.Linear(joint_dim, common_dim)

        # Scalar gate per modality
        self.depth_gate = nn.Linear(common_dim, 1)
        self.rgb_gate = nn.Linear(common_dim, 1)
        self.joint_gate = nn.Linear(common_dim, 1)

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(common_dim * 3, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, depth_features, rgb_features, joint_features):
        # Project to common dimension
        depth_features = self.depth_proj(depth_features)
        rgb_features = self.rgb_proj(rgb_features)
        joint_features = self.joint_proj(joint_features)

        # Compute scalar gates
        depth_weight = torch.sigmoid(self.depth_gate(depth_features))
        rgb_weight = torch.sigmoid(self.rgb_gate(rgb_features))
        joint_weight = torch.sigmoid(self.joint_gate(joint_features))

        # Apply gating
        depth_features = depth_weight * depth_features
        rgb_features = rgb_weight * rgb_features
        joint_features = joint_weight * joint_features

        # Concatenate gated features
        fused_features = torch.cat([depth_features, rgb_features, joint_features], dim=1)

        # Classify
        logits = self.classifier(fused_features)
        return logits

## Fusion Model Sanity Check

Instantiate the gated fusion model and run a quick dummy forward pass using encoder feature outputs to verify that the final output shape matches the number of classes.

In [11]:
fusion_model = GatedFusionModel().to(device)

with torch.no_grad():
    dummy_depth = torch.randn(2, DEPTH_FEATURE_DIM).to(device)
    dummy_rgb = torch.randn(2, RGB_FEATURE_DIM).to(device)
    dummy_joint = torch.randn(2, JOINT_FEATURE_DIM).to(device)

    logits = fusion_model(dummy_depth, dummy_rgb, dummy_joint)

print("Fusion output shape:", logits.shape)

Fusion output shape: torch.Size([2, 3])


## Training and Validation Loop (Frozen Encoders)

Train the gated fusion model using features extracted from pretrained encoders.

- Encoders are frozen (no gradient updates)
- Only fusion model parameters are trained
- Evaluate using accuracy and macro F1 score
- Save best model based on validation F1

In [13]:
# =========================
# Loss and optimizer
# =========================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(fusion_model.parameters(), lr=LEARNING_RATE)

best_val_f1 = 0.0
best_model_path = FUSION_CHECKPOINT_DIR / "best_gated_fusion_model.pth"

In [14]:
# =========================
# Training loop
# =========================
for epoch in range(NUM_EPOCHS):
    fusion_model.train()

    train_losses = []
    train_preds = []
    train_labels = []

    for batch in train_loader:
        rgb = batch["rgb"].to(device)
        depth = batch["depth"].to(device)
        joint = batch["joint"].to(device)
        labels = batch["label"].to(device)

        # -------------------------
        # Extract features (frozen encoders)
        # -------------------------
        with torch.no_grad():
            f_rgb = rgb_encoder(rgb)
            f_depth = depth_encoder(depth)
            f_joint = joint_encoder(joint)

        # -------------------------
        # Forward pass
        # -------------------------
        outputs = fusion_model(f_depth, f_rgb, f_joint)
        loss = criterion(outputs, labels)

        # -------------------------
        # Backprop
        # -------------------------
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # -------------------------
        # Metrics tracking
        # -------------------------
        train_losses.append(loss.item())
        preds = torch.argmax(outputs, dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    train_acc = (np.array(train_preds) == np.array(train_labels)).mean()
    train_f1 = f1_score(train_labels, train_preds, average="macro")

    # =========================
    # Validation
    # =========================
    fusion_model.eval()

    val_preds = []
    val_labels = []

    with torch.no_grad():
        for batch in val_loader:
            rgb = batch["rgb"].to(device)
            depth = batch["depth"].to(device)
            joint = batch["joint"].to(device)
            labels = batch["label"].to(device)

            f_rgb = rgb_encoder(rgb)
            f_depth = depth_encoder(depth)
            f_joint = joint_encoder(joint)

            outputs = fusion_model(f_depth, f_rgb, f_joint)
            preds = torch.argmax(outputs, dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_acc = (np.array(val_preds) == np.array(val_labels)).mean()
    val_f1 = f1_score(val_labels, val_preds, average="macro")

    # =========================
    # Logging
    # =========================
    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"Train Loss: {np.mean(train_losses):.4f}")
    print(f"Train Acc:  {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val Acc:    {val_acc:.4f} | Val F1:   {val_f1:.4f}")

    # =========================
    # Save best model
    # =========================
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(fusion_model.state_dict(), best_model_path)
        print("✅ Saved best model")


Epoch [1/15]
Train Loss: 0.0270
Train Acc:  0.9937 | Train F1: 0.9937
Val Acc:    0.9970 | Val F1:   0.9970
✅ Saved best model

Epoch [2/15]
Train Loss: 0.0243
Train Acc:  0.9942 | Train F1: 0.9942
Val Acc:    0.9970 | Val F1:   0.9970

Epoch [3/15]
Train Loss: 0.0217
Train Acc:  0.9939 | Train F1: 0.9940
Val Acc:    0.9970 | Val F1:   0.9970

Epoch [4/15]
Train Loss: 0.0227
Train Acc:  0.9944 | Train F1: 0.9944
Val Acc:    0.9960 | Val F1:   0.9961

Epoch [5/15]
Train Loss: 0.0234
Train Acc:  0.9943 | Train F1: 0.9943
Val Acc:    0.9975 | Val F1:   0.9975
✅ Saved best model

Epoch [6/15]
Train Loss: 0.0212
Train Acc:  0.9945 | Train F1: 0.9945
Val Acc:    0.9970 | Val F1:   0.9970

Epoch [7/15]
Train Loss: 0.0227
Train Acc:  0.9943 | Train F1: 0.9943
Val Acc:    0.9970 | Val F1:   0.9970

Epoch [8/15]
Train Loss: 0.0181
Train Acc:  0.9957 | Train F1: 0.9957
Val Acc:    0.9980 | Val F1:   0.9980
✅ Saved best model

Epoch [9/15]
Train Loss: 0.0181
Train Acc:  0.9953 | Train F1: 0.9953


## Test Evaluation

Load the best saved gated fusion model and evaluate it on the test set.

This section reports:

- test accuracy
- test macro F1
- confusion matrix

These results represent the final performance of the gated fusion experiment.

In [17]:
# =========================
# Load best model
# =========================
fusion_model.load_state_dict(torch.load(best_model_path, map_location=device))
fusion_model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for batch in test_loader:
        rgb = batch["rgb"].to(device)
        depth = batch["depth"].to(device)
        joint = batch["joint"].to(device)
        labels = batch["label"].to(device)

        f_rgb = rgb_encoder(rgb)
        f_depth = depth_encoder(depth)
        f_joint = joint_encoder(joint)

        outputs = fusion_model(f_depth, f_rgb, f_joint)
        preds = torch.argmax(outputs, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, test_preds, average="macro")
test_cm = confusion_matrix(test_labels, test_preds)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")
print("\nConfusion Matrix:")
print(test_cm)

Test Accuracy: 0.9898
Test Macro F1: 0.9898

Confusion Matrix:
[[709   9   2]
 [  4 716   0]
 [  7   0 713]]


## Condition-wise Test Evaluation

Evaluate test performance separately for each condition:

- uncover
- cover1
- cover2

This helps assess how well the gated fusion model performs under occlusion.

In [18]:
condition_preds = {"uncover": [], "cover1": [], "cover2": []}
condition_labels = {"uncover": [], "cover1": [], "cover2": []}

fusion_model.eval()

with torch.no_grad():
    for batch in test_loader:
        rgb = batch["rgb"].to(device)
        depth = batch["depth"].to(device)
        joint = batch["joint"].to(device)
        labels = batch["label"].to(device)
        conditions = batch["condition"]

        f_rgb = rgb_encoder(rgb)
        f_depth = depth_encoder(depth)
        f_joint = joint_encoder(joint)

        outputs = fusion_model(f_depth, f_rgb, f_joint)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        labels_np = labels.cpu().numpy()

        for i, cond in enumerate(conditions):
            condition_preds[cond].append(preds[i])
            condition_labels[cond].append(labels_np[i])

for cond in ["uncover", "cover1", "cover2"]:
    acc = accuracy_score(condition_labels[cond], condition_preds[cond])
    f1 = f1_score(condition_labels[cond], condition_preds[cond], average="macro")
    print(f"{cond} -> Accuracy: {acc:.4f}, Macro F1: {f1:.4f}")

uncover -> Accuracy: 0.9931, Macro F1: 0.9931
cover1 -> Accuracy: 0.9903, Macro F1: 0.9903
cover2 -> Accuracy: 0.9861, Macro F1: 0.9861


## Save Experiment Results

Save key outputs of the experiment for reproducibility and comparison:

- test metrics
- condition-wise metrics
- configuration used
- model checkpoint reference

This allows consistent comparison across different fusion strategies.

In [19]:
import json
from datetime import datetime

# =========================
# Collect condition-wise metrics
# =========================
condition_metrics = {}

for cond in ["uncover", "cover1", "cover2"]:
    acc = accuracy_score(condition_labels[cond], condition_preds[cond])
    f1 = f1_score(condition_labels[cond], condition_preds[cond], average="macro")

    condition_metrics[cond] = {
        "accuracy": float(acc),
        "macro_f1": float(f1)
    }

# =========================
# Build results dictionary
# =========================
results = {
    "experiment": "gated_fusion",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "config": {
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "common_feature_dim": COMMON_FEATURE_DIM,
        "dropout": DROPOUT,
    },

    "metrics": {
        "test_accuracy": float(test_acc),
        "test_macro_f1": float(test_f1),
        "confusion_matrix": test_cm.tolist()
    },

    "condition_metrics": condition_metrics,

    "paths": {
        "model_checkpoint": str(best_model_path),
    }
}

# =========================
# Save results
# =========================
results_path = FUSION_CHECKPOINT_DIR / "gated_fusion_results.json"

with open(results_path, "w") as f:
    json.dump(results, f, indent=4)

print("Results saved to:", results_path)

Results saved to: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\checkpoints_fusion_gated\gated_fusion_results.json
